# Apriori Principle - Starter Notebook
Implements the Apriori algorithm: any subset of a frequent itemset must also be frequent (anti-monotone property).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

In [ ]:
transactions = [
    {'A','B','C'},
    {'A','B'},
    {'A','C'},
    {'B','C','D'},
    {'A','B','C','D'},
    {'B','D'},
    {'A','C','D'},
    {'A','B','D'},
    {'C','D'},
    {'A','B','C','D'}
]
N = len(transactions)
MIN_SUPPORT = 0.4
print(f'Transactions: {N}, min_support: {MIN_SUPPORT}')

In [ ]:
def support(itemset, transactions):
    return sum(1 for t in transactions if itemset <= t) / len(transactions)

def apriori(transactions, min_support):
    items = sorted({item for t in transactions for item in t})
    L = {}
    # Generate C1
    L1 = {frozenset([i]): support(frozenset([i]), transactions)
          for i in items if support(frozenset([i]), transactions) >= min_support}
    L.update(L1)
    k = 2
    Lk_prev = list(L1.keys())
    while Lk_prev:
        # Candidate generation: join step
        Ck = set()
        for a, b in combinations(Lk_prev, 2):
            candidate = a | b
            if len(candidate) == k:
                # Pruning: all (k-1)-subsets must be frequent
                if all(frozenset(s) in L for s in combinations(candidate, k-1)):
                    Ck.add(candidate)
        Lk = {c: support(c, transactions) for c in Ck
              if support(c, transactions) >= min_support}
        L.update(Lk)
        Lk_prev = list(Lk.keys())
        k += 1
    return L

freq = apriori(transactions, MIN_SUPPORT)
for fs, sup in sorted(freq.items(), key=lambda x: (len(x[0]), -x[1])):
    print(f'  {set(fs)}  |  support = {sup:.2f}')

In [ ]:
# Demonstrate Apriori anti-monotone property
print('=== Apriori Anti-monotone Principle ===')
test_set = frozenset({'A', 'B', 'C'})
sup_test = support(test_set, transactions)
print(f'Support of {set(test_set)}: {sup_test:.2f}')
print('All subsets must have support >= this value:')
for k in range(1, len(test_set)):
    for s in combinations(test_set, k):
        fs = frozenset(s)
        print(f'  {set(fs)}: support={support(fs, transactions):.2f}')

In [ ]:
# Bar chart: support by itemset size
from collections import defaultdict
by_size = defaultdict(list)
for fs, sup in freq.items():
    by_size[len(fs)].append(sup)

sizes = sorted(by_size.keys())
avg_sups = [np.mean(by_size[s]) for s in sizes]

plt.figure(figsize=(6, 4))
plt.bar([str(s) for s in sizes], avg_sups, color='coral', edgecolor='k')
plt.xlabel('Itemset size (k)')
plt.ylabel('Average Support')
plt.title('Apriori: Average Support by Itemset Size')
plt.tight_layout()
plt.show()